# Data Profiling — [primary dataset name]

**Purpose:** describe the dataset's shape, distributions, gaps, and anomalies so we know what we have before building any pipeline on it.

**Input:** [path or URL to data]
**Output:** a list of findings written into `docs/data-quality-audit.md`.

In [37]:
!pip install earthengine-api geemap

In [38]:
import ee

ee.Authenticate()

ee.Initialize(project='ee-jineshnarendrajain')
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Authenticate (run once)
ee.Authenticate()

# Initialize Earth Engine
ee.Initialize()

In [ ]:
##4/1AeoWuM9BZfq3rPvE3PK2P1YrqIzy_BLrm5wwDwQW7STucA-mwVeqV75hcDQ

SyntaxError: invalid decimal literal (1233888375.py, line 1)

In [40]:
manhattan = ee.Geometry.Rectangle([-74.03, 40.70, -73.90, 40.88])

In [41]:
# -------------------------------
# 1. IMPORTS + INIT
# -------------------------------
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-jineshnarendrajain')





In [42]:
# -------------------------------
# 2. STUDY AREA (REAL MANHATTAN)
# -------------------------------
nyc = ee.FeatureCollection("TIGER/2018/Counties")
manhattan = nyc.filter(ee.Filter.eq('NAME', 'New York')).geometry()




In [43]:
# -------------------------------
# 3. LOAD LANDSAT (LST)
# -------------------------------
landsat = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(manhattan) \
    .filterDate('2018-06-01', '2018-08-31') \
    .filter(ee.Filter.lt('CLOUD_COVER', 20))



In [44]:
# -------------------------------
# 4. EXTRACT LST
# -------------------------------
def get_lst(image):
    lst = image.select('ST_B10') \
        .multiply(0.00341802) \
        .add(149.0) \
        .subtract(273.15) \
        .rename('LST')
    return lst.copyProperties(image, ['system:time_start'])

lst_collection = landsat.map(get_lst)
lst_mean = lst_collection.mean().clip(manhattan)


In [45]:
# -------------------------------
# 5. LOAD SENTINEL-2 (NDVI)
# -------------------------------
sentinel = ee.ImageCollection("COPERNICUS/S2_SR") \
    .filterBounds(manhattan) \
    .filterDate('2018-06-01', '2018-08-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

In [46]:
# -------------------------------
# 6. CLOUD MASK (SCL BASED)
# -------------------------------
def mask_s2_clouds(image):
    scl = image.select('SCL')

    mask = scl.neq(3) \
        .And(scl.neq(8)) \
        .And(scl.neq(9)) \
        .And(scl.neq(10)) \
        .And(scl.neq(11))

    return image.updateMask(mask)


sentinel_clean = sentinel.map(mask_s2_clouds)

In [47]:
# -------------------------------
# 7. COMPUTE NDVI
# -------------------------------
def get_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi.copyProperties(image, ['system:time_start'])

ndvi_collection = sentinel_clean.map(get_ndvi)
ndvi_mean = ndvi_collection.median().clip(manhattan)


In [48]:
# -------------------------------
# 8. VISUALIZATION
# -------------------------------
Map = geemap.Map(center=[40.78, -73.97], zoom=12)

lst_vis = {
    'min': 20,
    'max': 40,
    'palette': ['blue', 'green', 'yellow', 'red']
}

ndvi_vis = {
    'min': 0.1,
    'max': 0.8,
    'palette': ['brown', 'yellow', 'green']
}

Map.addLayer(lst_mean, lst_vis, 'LST (Summer)')
Map.addLayer(ndvi_mean, ndvi_vis, 'NDVI (Clean)')
Map

Map(center=[40.78, -73.97], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [ ]:
# -------------------------------
# 9. BASIC STATISTICS (LST)
# -------------------------------

lst_stats = lst_mean.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=manhattan,
    scale=30,
    maxPixels=1e9
)

print("LST stats:", lst_stats.getInfo())

LST stats: {'LST_max': 49.20172034000001, 'LST_mean': 30.732315518857664, 'LST_min': 10.331996900000036}


In [ ]:
# -------------------------------
# 10. NDVI STATISTICS
# -------------------------------

ndvi_stats = ndvi_mean.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=manhattan,
    scale=10,
    maxPixels=1e9
)

print("NDVI stats:", ndvi_stats.getInfo())

NDVI stats: {'NDVI_max': 0.6994929909706116, 'NDVI_mean': 0.0916949249737827, 'NDVI_min': -0.27076125144958496}
